# Text Privacy Classifier with LSTM

This notebook trains a simple text classifier for the Home AI project. The goal is to classify speech-to-text output as either `private` or `non_private` before deciding whether it is safe to offload a command.

This is a first baseline model. It uses a small LSTM instead of an image model or transformer so the code stays readable and practical for early experiments.

## 1. Imports and setup

This cell imports the libraries used in the notebook and fixes the random seed so the split and training behavior are easier to reproduce.

In [ ]:
import json
import random
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 2. Dataset loading placeholder

The real privacy dataset will be added later. For now, this placeholder dataframe shows the expected format: one `text` column and one `label` column. Labels should be `private` or `non_private`.

In [ ]:
# TODO: Replace this placeholder with the real dataset file when it is available.
# Example:
# df = pd.read_csv("data/privacy_text_dataset.csv")

placeholder_data = [
    {"text": "What is the weather today", "label": "non_private"},
    {"text": "Play relaxing music in the living room", "label": "non_private"},
    {"text": "Turn on the kitchen lights", "label": "non_private"},
    {"text": "Set a timer for ten minutes", "label": "non_private"},
    {"text": "Tell me a fun fact about space", "label": "non_private"},
    {"text": "Increase the thermostat by two degrees", "label": "non_private"},
    {"text": "Add milk and bread to my shopping list", "label": "non_private"},
    {"text": "What time is sunset tonight", "label": "non_private"},
    {"text": "Read my latest bank balance", "label": "private"},
    {"text": "Send my medical report to the doctor", "label": "private"},
    {"text": "What is my home address", "label": "private"},
    {"text": "Text Sarah my door code", "label": "private"},
    {"text": "Read the verification code from my messages", "label": "private"},
    {"text": "Call my insurance provider about my claim", "label": "private"},
    {"text": "Open my private email inbox", "label": "private"},
    {"text": "Remind me to take my blood pressure medicine", "label": "private"},
    {"text": "Start the robot vacuum", "label": "non_private"},
    {"text": "Dim the bedroom lights", "label": "non_private"},
    {"text": "Read my salary document", "label": "private"},
    {"text": "Send my passport number to the airline", "label": "private"},
    {"text": "Pause the TV", "label": "non_private"},
    {"text": "Order more coffee beans", "label": "non_private"},
    {"text": "Find my national ID number", "label": "private"},
    {"text": "Read the last message from my lawyer", "label": "private"},
]

df = pd.DataFrame(placeholder_data)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())
print("\nLabel counts:")
print(df["label"].value_counts())

## 3. Dataset cleaning placeholder

This section keeps the cleaning simple. The real dataset may need extra work for missing values, inconsistent labels, brackets, commas, or unusual formatting.

In [ ]:
df_clean = df.copy()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[\[\]\(\){}]", " ", text)
    text = re.sub(r",", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def clean_label(label):
    label = str(label).strip().lower()
    label = label.replace("-", "_").replace(" ", "_")
    return label

df_clean = df_clean.dropna(subset=["text", "label"])
df_clean["text"] = df_clean["text"].apply(clean_text)
df_clean["label"] = df_clean["label"].apply(clean_label)

# TODO: Add project-specific cleaning rules after inspecting the real dataset.
valid_labels = {"private", "non_private"}
df_clean = df_clean[df_clean["label"].isin(valid_labels)]
df_clean = df_clean.drop_duplicates(subset=["text", "label"]).reset_index(drop=True)

print("Clean shape:", df_clean.shape)
display(df_clean.head())
print("\nClean label counts:")
print(df_clean["label"].value_counts())

## 4. Label encoding

The model needs numeric labels. We treat `private` as the positive class because missing private text is the most dangerous mistake.

In [ ]:
LABEL_TO_ID = {
    "non_private": 0,
    "private": 1,
}

ID_TO_LABEL = {
    0: "non_private",
    1: "private",
}

df_clean["label_id"] = df_clean["label"].map(LABEL_TO_ID)

print("private is the positive class with label id 1")
display(df_clean[["text", "label", "label_id"]].head())

## 5. Train, validation, and test split

This cell creates a reproducible split. The vocabulary will be built only from the training text to avoid using validation or test information during preprocessing.

In [ ]:
df_shuffled = df_clean.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

train_end = int(0.70 * len(df_shuffled))
val_end = int(0.85 * len(df_shuffled))

train_df = df_shuffled.iloc[:train_end].reset_index(drop=True)
val_df = df_shuffled.iloc[train_end:val_end].reset_index(drop=True)
test_df = df_shuffled.iloc[val_end:].reset_index(drop=True)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain labels:")
print(train_df["label"].value_counts())
print("\nValidation labels:")
print(val_df["label"].value_counts())
print("\nTest labels:")
print(test_df["label"].value_counts())

## 6. Tokenization and vocabulary

This baseline uses a simple whitespace tokenizer. The vocabulary maps words from the training text to token IDs, with special tokens for padding and unknown words.

In [ ]:
def tokenize(text):
    return str(text).split()

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

counter = Counter()
for text in train_df["text"]:
    counter.update(tokenize(text))

MAX_VOCAB_SIZE = 2000

vocab = {
    PAD_TOKEN: 0,
    UNK_TOKEN: 1,
}

for word, count in counter.most_common(MAX_VOCAB_SIZE - 2):
    vocab[word] = len(vocab)

print("Vocabulary size:", len(vocab))
print("\nFirst 20 vocabulary items:")
for i, (word, idx) in enumerate(vocab.items()):
    if i == 20:
        break
    print(word, "->", idx)

## 7. Encode text as token IDs

Each text sample is converted into token IDs. Long texts are truncated to a fixed maximum length, and short texts are padded later inside each batch.

In [ ]:
MAX_LEN = 40

def encode_text(text, vocab, max_len=MAX_LEN):
    tokens = tokenize(text)
    ids = [vocab.get(token, vocab[UNK_TOKEN]) for token in tokens]
    ids = ids[:max_len]

    if len(ids) == 0:
        ids = [vocab[UNK_TOKEN]]

    return ids

def encode_dataframe(dataframe, vocab):
    return [
        (encode_text(row["text"], vocab), row["label_id"])
        for _, row in dataframe.iterrows()
    ]

train_encoded = encode_dataframe(train_df, vocab)
val_encoded = encode_dataframe(val_df, vocab)
test_encoded = encode_dataframe(test_df, vocab)

print("Original text:")
print(train_df.loc[0, "text"])
print("\nEncoded text:")
print(train_encoded[0][0])
print("\nLabel:")
print(train_encoded[0][1], ID_TO_LABEL[train_encoded[0][1]])

## 8. Dataset and DataLoader

This PyTorch dataset returns encoded text and labels. The collate function pads each batch to the longest text in that batch, similar to the KAUST RNN/LSTM lab style.

In [ ]:
class PrivacyTextDataset(Dataset):
    def __init__(self, encoded_data):
        self.encoded_data = encoded_data

    def __len__(self):
        return len(self.encoded_data)

    def __getitem__(self, idx):
        token_ids, label = self.encoded_data[idx]
        token_ids = torch.tensor(token_ids, dtype=torch.long)
        label = torch.tensor(label, dtype=torch.float32)
        return token_ids, label

def collate_batch(batch):
    texts, labels = zip(*batch)

    lengths = torch.tensor(
        [len(text) for text in texts],
        dtype=torch.long
    )

    padded_texts = nn.utils.rnn.pad_sequence(
        texts,
        batch_first=True,
        padding_value=vocab[PAD_TOKEN]
    )

    labels = torch.stack(labels)

    return padded_texts, labels, lengths

BATCH_SIZE = 8

train_loader = DataLoader(
    PrivacyTextDataset(train_encoded),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_batch
)

val_loader = DataLoader(
    PrivacyTextDataset(val_encoded),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch
)

test_loader = DataLoader(
    PrivacyTextDataset(test_encoded),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch
)

print("Number of training batches:", len(train_loader))
print("Number of validation batches:", len(val_loader))
print("Number of test batches:", len(test_loader))

## 9. LSTM model definition

The model uses an embedding layer, one LSTM layer, dropout, and a final linear layer. It returns one logit per text sample for binary classification.

In [ ]:
class LSTMPrivacyClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, dropout_rate=0.3):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=vocab[PAD_TOKEN]
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, text, lengths):
        embedded = self.embedding(text)

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        packed_output, (hidden, cell) = self.lstm(packed)
        final_hidden = hidden[-1]
        final_hidden = self.dropout(final_hidden)

        logits = self.fc(final_hidden)
        return logits.squeeze(1)

## 10. Training setup

This cell sets the model size, loss function, optimizer, learning rate, and number of epochs. `BCEWithLogitsLoss` fits this binary classification setup.

In [ ]:
VOCAB_SIZE = len(vocab)
EMBEDDING_DIM = 64
HIDDEN_DIM = 64
DROPOUT_RATE = 0.3
EPOCHS = 8
LEARNING_RATE = 0.001

model = LSTMPrivacyClassifier(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    dropout_rate=DROPOUT_RATE
).to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

print(model)

## 11. Metrics helper

These metrics treat `private` as the positive class. Recall is especially important because a false negative means private text was predicted as non-private.

In [ ]:
def calculate_binary_metrics(labels, predictions):
    labels = np.array(labels)
    predictions = np.array(predictions)

    tp = int(((labels == 1) & (predictions == 1)).sum())
    tn = int(((labels == 0) & (predictions == 0)).sum())
    fp = int(((labels == 0) & (predictions == 1)).sum())
    fn = int(((labels == 1) & (predictions == 0)).sum())

    accuracy = (tp + tn) / max(tp + tn + fp + fn, 1)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = (2 * precision * recall) / max(precision + recall, 1e-8)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }

## 12. Training and validation loop

The model trains for a few epochs and validates after each epoch. We track loss, accuracy, precision, recall, and F1 score.

In [ ]:
def train_one_epoch(model, dataloader, optimizer, criterion):
    model.train()

    total_loss = 0.0
    total = 0
    all_labels = []
    all_predictions = []

    for text, labels, lengths in dataloader:
        text = text.to(device)
        labels = labels.to(device).float()

        optimizer.zero_grad()
        logits = model(text, lengths)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total += batch_size

        probabilities = torch.sigmoid(logits)
        predictions = probabilities >= 0.5

        all_labels.extend(labels.cpu().numpy().astype(int).tolist())
        all_predictions.extend(predictions.cpu().numpy().astype(int).tolist())

    metrics = calculate_binary_metrics(all_labels, all_predictions)
    metrics["loss"] = total_loss / total

    return metrics

def evaluate(model, dataloader, criterion):
    model.eval()

    total_loss = 0.0
    total = 0
    all_labels = []
    all_predictions = []

    with torch.no_grad():
        for text, labels, lengths in dataloader:
            text = text.to(device)
            labels = labels.to(device).float()

            logits = model(text, lengths)
            loss = criterion(logits, labels)

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total += batch_size

            probabilities = torch.sigmoid(logits)
            predictions = probabilities >= 0.5

            all_labels.extend(labels.cpu().numpy().astype(int).tolist())
            all_predictions.extend(predictions.cpu().numpy().astype(int).tolist())

    metrics = calculate_binary_metrics(all_labels, all_predictions)
    metrics["loss"] = total_loss / total

    return metrics

## 13. Train the model

This cell runs training and prints a compact summary after every epoch.

In [ ]:
history = []

print("Training LSTM privacy classifier")
print("=" * 70)

for epoch in range(EPOCHS):
    train_metrics = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion
    )

    val_metrics = evaluate(
        model,
        val_loader,
        criterion
    )

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "train_precision": train_metrics["precision"],
        "train_recall": train_metrics["recall"],
        "train_f1": train_metrics["f1"],
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_precision": val_metrics["precision"],
        "val_recall": val_metrics["recall"],
        "val_f1": val_metrics["f1"],
    })

    print(
        f"Epoch {epoch + 1:02d} | "
        f"Train Loss: {train_metrics['loss']:.4f} | "
        f"Val Loss: {val_metrics['loss']:.4f} | "
        f"Val Acc: {val_metrics['accuracy']:.4f} | "
        f"Val Precision: {val_metrics['precision']:.4f} | "
        f"Val Recall: {val_metrics['recall']:.4f} | "
        f"Val F1: {val_metrics['f1']:.4f}"
    )

history_df = pd.DataFrame(history)
display(history_df)

## 14. Evaluation on the test set

This final evaluation checks the model on the held-out test split. False negatives are the most dangerous error because private text would be treated as non-private.

In [ ]:
test_metrics = evaluate(model, test_loader, criterion)

print("Test metrics")
print("=" * 70)
print(f"Loss:      {test_metrics['loss']:.4f}")
print(f"Accuracy:  {test_metrics['accuracy']:.4f}")
print(f"Precision: {test_metrics['precision']:.4f}")
print(f"Recall:    {test_metrics['recall']:.4f}")
print(f"F1 score:  {test_metrics['f1']:.4f}")

print("\nConfusion matrix values")
print("=" * 70)
print("True Positive  (private predicted private):", test_metrics["tp"])
print("True Negative  (non-private predicted non-private):", test_metrics["tn"])
print("False Positive (non-private predicted private):", test_metrics["fp"])
print("False Negative (private predicted non-private):", test_metrics["fn"])

print("\nPrivacy note")
print("False Negative: private text predicted as non-private. This is the most dangerous error.")
print("False Positive: non-private text predicted as private. This is less dangerous but may reduce usability.")

## 15. Plot training curves

These plots show loss, precision, recall, and F1 over epochs. For privacy detection, recall for the private class is especially important.

In [ ]:
epochs = history_df["epoch"]

plt.figure(figsize=(8, 5))
plt.plot(epochs, history_df["train_loss"], label="Train Loss", marker="o")
plt.plot(epochs, history_df["val_loss"], label="Validation Loss", marker="o")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss over Epochs")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(epochs, history_df["val_precision"], label="Validation Precision", marker="o")
plt.xlabel("Epoch")
plt.ylabel("Precision")
plt.title("Precision over Epochs")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(epochs, history_df["val_recall"], label="Validation Recall", marker="o")
plt.xlabel("Epoch")
plt.ylabel("Recall")
plt.title("Recall over Epochs")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(epochs, history_df["val_f1"], label="Validation F1", marker="o")
plt.xlabel("Epoch")
plt.ylabel("F1 Score")
plt.title("F1 Score over Epochs")
plt.grid(True)
plt.legend()
plt.show()

## 16. Save the model and vocabulary

This cell saves the trained model weights and the vocabulary mapping. The vocabulary is needed later to encode new text in the same way.

In [ ]:
model_dir = Path("model")
output_dir = model_dir / "artifacts" if model_dir.exists() else Path("artifacts")
output_dir.mkdir(parents=True, exist_ok=True)

model_path = output_dir / "text_privacy_lstm.pt"
vocab_path = output_dir / "text_privacy_vocab.json"
config_path = output_dir / "text_privacy_config.json"

torch.save(model.state_dict(), model_path)

with open(vocab_path, "w", encoding="utf-8") as f:
    json.dump(vocab, f, indent=2)

config = {
    "label_to_id": LABEL_TO_ID,
    "id_to_label": ID_TO_LABEL,
    "max_len": MAX_LEN,
    "embedding_dim": EMBEDDING_DIM,
    "hidden_dim": HIDDEN_DIM,
    "dropout_rate": DROPOUT_RATE,
    "threshold": 0.5,
}

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print("Saved model to:", model_path)
print("Saved vocabulary to:", vocab_path)
print("Saved config to:", config_path)

## 17. Next steps

- Replace the placeholder dataframe with the real Home AI privacy dataset.
- Improve preprocessing after inspecting real examples.
- Tune the prediction threshold to reduce false negatives.
- Test the model on realistic Home AI speech-to-text examples.
- Later, compress or simplify the model for Raspberry Pi deployment.